# 06 — Groq LLM Testing

Purpose: test prompt design and generation quality against Groq directly,
using the already-built retrieval from notebook 05 / `retriever.py`.
Iterate on `SYSTEM_PROMPT` in `src/rag/llm_client.py` here before wiring
it into the Streamlit chat page.


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import os
import yaml
from dotenv import load_dotenv
from groq import Groq

load_dotenv(Path.cwd().parent / ".env")

with open("../config/config.yaml") as f:
    config = yaml.safe_load(f)

genai_config = config["genai"]
client = Groq(api_key=os.environ["GROQ_API_KEY"])

## 1. Confirm the configured model is available

In [2]:
# List models Groq currently exposes -- worth checking since available
# models rotate; if genai_config["model"] below 404s, update config.yaml
# to whatever's currently listed.
models = client.models.list()
for m in models.data:
    print(m.id)


openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3-turbo
groq/compound-mini
whisper-large-v3
allam-2-7b
openai/gpt-oss-120b
llama-3.1-8b-instant
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-safeguard-20b
llama-3.3-70b-versatile
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
qwen/qwen3.6-27b
groq/compound


## 2. Bare generation test (no retrieval yet)

In [3]:
response = client.chat.completions.create(
    model=genai_config["model"],
    messages=[{"role": "user", "content": "In one sentence, what does stock volatility measure?"}],
    temperature=genai_config["temperature"],
    max_tokens=100,
)
print(response.choices[0].message.content)


Stock volatility measures the degree of uncertainty or fluctuation in a stock's price over a given period of time, reflecting the amount of risk or potential for significant price changes.


## 3. Full RAG-grounded generation using the production function


In [ ]:
from src.rag.llm_client import answer_query

CONFIG_PATH = str(Path.cwd().parent / "config" / "config.yaml")

sample_ticker = "TATACOMM"   # Change to a ticker for which you have built a FAISS index

test_questions = [
    "why is this stock considered risky right now?",
    "has recent news been positive or negative for this company?",
    "what's the 20-day volatility trend?",
]

for q in test_questions:
    print(f"\nQ: {q}")
    print(
        "A:",
        answer_query(
            ticker=sample_ticker,
            query=q,
            config_path=CONFIG_PATH
        )
    )

## 4. Iterate on the system prompt

Copy `SYSTEM_PROMPT` from `src/rag/llm_client.py` here, tweak it, and
re-run generation directly (bypassing the production function) to A/B
different phrasing before committing a change back to the source file.


In [ ]:
from src.rag.retriever import retrieve_relevant_chunks

CONFIG_PATH = str(Path.cwd().parent / "config" / "config.yaml")

def test_prompt(system_prompt, ticker, query):
    chunks = retrieve_relevant_chunks(
        ticker=ticker,
        query=query,
        config_path=CONFIG_PATH
    )

    context = "\n".join(f"- {c}" for c in chunks) if chunks else "No context available."

    user_content = f"""Ticker: {ticker}

Retrieved context:
{context}

User question:
{query}
"""

    resp = client.chat.completions.create(
        model=genai_config["model"],
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
        ],
        temperature=genai_config["temperature"],
        max_tokens=genai_config["max_tokens"],
    )

    return resp.choices[0].message.content

alt_system_prompt = """
You are EquiRisk's assistant. Answer using ONLY the provided context.
Keep answers under 4 sentences. Never give direct buy/sell advice.
"""

print(
    test_prompt(
        alt_system_prompt,
        sample_ticker,
        "why is this stock considered risky right now?"
    )
)